# Experiment 1: Your first AI agent (Hugging Face edition)

**Fact Checker and Tweet Bot on free open-source models from Hugging Face.** No API key needed.

**Analogy:** OpenAI, Gemini and Groq are restaurants: you order through a waiter (the API key) and the kitchen is theirs. Here we cook at home. Hugging Face is the free recipe library, the Qwen model is the recipe, Ollama is the cook, and Colab's free GPU is the stove. The Agents SDK places the order the same way; it just sends it to your own kitchen.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

## Step 1: Switch Colab to the free GPU

The model will run inside your own Colab, so it needs Colab's free T4 GPU. Menu: Runtime → Change runtime type → T4 GPU → Save. Then run this cell to confirm.

In [ ]:
# STEP 1: Check that Colab gave you a free GPU
# Runtime > Change runtime type > T4 GPU > Save, then run this cell
!nvidia-smi

## Step 2: Install the Agents SDK

Installs the exact SDK version used in the course, plus helper libraries. Same as the OpenAI and Groq labs.

In [ ]:
# STEP 2: Install the Agents SDK
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic

## Step 3: Install Ollama, the model runner

Ollama is a free program that downloads open-source models and serves them at an OpenAI-style address, so the Agents SDK can talk to them without any code change.

In [ ]:
# STEP 3: Install Ollama (free program that runs open-source models)
# zstd unpacks Ollama's download; pciutils and lshw help it find the GPU
!sudo apt-get -qq update
!sudo apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## Step 4: Start the Ollama server

Starts Ollama in the background. It keeps running while this notebook is open. If Colab restarts, run this step again (the install and the downloads are kept).

In [ ]:
# STEP 4: Start Ollama in the background
import subprocess, time, requests

ollama_log = open("ollama.log", "w")
ollama_server = subprocess.Popen(["ollama", "serve"], stdout = ollama_log, stderr = subprocess.STDOUT)
time.sleep(5)                                        # give it a few seconds to start

print(requests.get("http://localhost:11434").text)   # should print: Ollama is running

## Step 5: Download two open-source models from Hugging Face

hf.co/... tells Ollama to fetch the model straight from Hugging Face. Both are Qwen 2.5 models from Alibaba's Qwen team, in GGUF format (a compressed copy that fits a free GPU).

In [ ]:
# STEP 5: Download two open-source models from Hugging Face
# Small and fast, about 2 GB   (replaces gpt-4.1-mini / gpt-oss-20b)
!ollama pull hf.co/bartowski/Qwen2.5-3B-Instruct-GGUF:Q4_K_M

# Bigger and smarter, about 4.7 GB (replaces the larger GPT models / gpt-oss-120b)
!ollama pull hf.co/bartowski/Qwen2.5-7B-Instruct-GGUF:Q4_K_M

## Step 6: Connect the Agents SDK to your model server

Same OpenAI library, different address: requests now go to Ollama inside your Colab, not to OpenAI. No API key is needed because the model runs on your own machine; the library just needs some text in api_key.

In [ ]:
# STEP 6: Point the Agents SDK at Ollama instead of OpenAI
from openai import OpenAI, AsyncOpenAI
from IPython.display import display, Markdown
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

LOCAL_BASE_URL = "http://localhost:11434/v1"   # Ollama's OpenAI-compatible address

# No real key needed: any text works, because nobody is billing you
hf_client = AsyncOpenAI(api_key = "ollama", base_url = LOCAL_BASE_URL)

# Traces are uploaded to OpenAI's dashboard and need an OpenAI key, so switch them off
set_tracing_disabled(True)

print("Connected to the local Hugging Face models.")

## Step 7: Choose the models and list the available ones

hf() wraps a model name so the Agents SDK sends it to Ollama exactly as written. The list shows every model you downloaded.

In [ ]:
# STEP 7: Choose the models and see which ones are ready
HF_MODEL     = "hf.co/bartowski/Qwen2.5-3B-Instruct-GGUF:Q4_K_M"   # small and fast
HF_BIG_MODEL = "hf.co/bartowski/Qwen2.5-7B-Instruct-GGUF:Q4_K_M"   # bigger and smarter

def hf(model_name):
    """Wraps a model name so the Agents SDK sends it to our local Ollama server."""
    return OpenAIChatCompletionsModel(model = model_name, openai_client = hf_client)

sync_client = OpenAI(api_key = "ollama", base_url = LOCAL_BASE_URL)
for m in sync_client.models.list().data:
    print(m.id)

## Step 8: Write the agent's instructions

Exactly the same instructions as the OpenAI and Groq labs. Only the model behind the agent changes.

In [ ]:
from agents import Agent

fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
3. if the input is not a fact checking question and its a general question donot use any verdict prefix, give just a plane response
"""

## Step 9: Create the Fact Checker agent

Name, instructions, model: the three parts of every agent.

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",                    # badge name
                           instructions = fact_checker_instructions, # job description
                           model = hf(HF_MODEL))                     # which brain (open-source, in your Colab)

print(f"Agent '{fact_checker_agent.name}' created successfully!")

## Step 10: Helper to show formatted answers

Shows the agent's reply as formatted text.

In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

## Step 11: Run the agent

Runner.run sends the statement to the agent and waits for the final answer. The first run takes 10–20 seconds while the model loads onto the GPU; later runs are quicker.

In [ ]:
from agents import Runner

statement = "The Great Wall of China is visible from space with the naked eye."
print_markdown(f"Asking the Fact Checker to verify: '{statement}'")

response = await Runner.run(
    starting_agent = fact_checker_agent,   # WHICH agent to run
    input = statement                      # WHAT to give it
)

print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)      # the agent's final answer

## Step 12: See how many tokens were used

Tokens still matter even when they are free: they decide speed and how much text fits in the model's memory window.

In [ ]:
usage = response.context_wrapper.usage
print(f"Requests: {usage.requests} | Input tokens: {usage.input_tokens} | "
      f"Output tokens: {usage.output_tokens} | Total: {usage.total_tokens}")

## Step 13: Test its memory

Each Runner.run starts fresh. The agent has no memory of the previous call.

In [ ]:
check_recall_statement = "What did we discuss in the last message?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)

## Step 14: Test its memory again

Same test, different wording. Still no memory.

In [ ]:
check_recall_statement = "Do you remmember the fact check statement/question?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)

## Step 15: Try the bigger model

Swap only the model: 3B → 7B. Everything else stays the same.

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = hf(HF_BIG_MODEL))

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)

## Step 16: Tune the small model for short, steady answers

The OpenAI/Groq lab used reasoning effort here. Qwen 2.5 has no thinking dial, so we use the two dials every model has: temperature = 0 (same answer every time, no creativity) and max_tokens = 60 (keep it short).

In [ ]:
from agents import ModelSettings

fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = hf(HF_MODEL),
                           model_settings = ModelSettings(temperature = 0, max_tokens = 60))

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)

## Step 17: Practice: build a Tweet Bot

Same Tweet Bot as before, on the bigger open-source model.

In [ ]:
tweet_bot_instructions = """
Instruction:
Write a short, engaging tweet (≤280 characters) about the given topic. The tweet must
include at least one relevant emoji and one relevant hashtag.

Context:
You are **TweetBot**, a witty and creative social-media expert known for crafting viral tweets.

Input:
A topic provided by the user.

Output:
A single tweet (≤280 characters) about the topic, containing at least one relevant emoji
and one relevant hashtag.
"""

tweet_bot_agent = Agent(name = "Tweet Bot",
                        instructions = tweet_bot_instructions,
                        model = hf(HF_BIG_MODEL))

topic = "The future of renewable energy"
tweet_response = await Runner.run(starting_agent = tweet_bot_agent, input = topic)
print_markdown(tweet_response.final_output)

## Step 18: Run an open-source model on Hugging Face's cloud (optional)

Skip the download and call the same kind of model on Hugging Face's servers. Needs a Hugging Face token (permission: Make calls to Inference Providers) saved in Colab Secrets as HF_TOKEN, and credits on that account: free accounts get no included credits, PRO gets $2 a month. Good for a trainer demo, not for a whole class.

In [ ]:
# OPTIONAL: same agent, open-source model on Hugging Face's cloud (needs credits)
from google.colab import userdata

hf_cloud_client = AsyncOpenAI(api_key = userdata.get("HF_TOKEN"),
                              base_url = "https://router.huggingface.co/v1")

cloud_fact_checker = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = OpenAIChatCompletionsModel(model = "Qwen/Qwen2.5-7B-Instruct",
                                                              openai_client = hf_cloud_client))

statement = "The Great Wall of China is visible from space with the naked eye."
response = await Runner.run(starting_agent = cloud_fact_checker, input = statement)
print_markdown(response.final_output)